# Pancancer enrichment analysis step 10: Make figures from GSEApy prerank data

## Setup

In [1]:
import cptac
import cptac.utils as ut
import os
import datetime
import altair as alt
import numpy as np
import pandas as pd
import operator
import IPython.display

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# Step 1 output
STEP1_DIR = "step1_outputs"
STEP1_FILE = "tumor_change_20200529_195104_10000_perm.tsv"
STEP1_FILE_PATH = os.path.join(STEP1_DIR, STEP1_FILE)

# Step 9 output
STEP9_DIR = "step9_outputs"
STEP9_FILE = "enrichment_gseapy_prerank_20200618_122915_3_perms_asc_True_wst_1.tsv"
STEP9_FILE_PATH = os.path.join(STEP9_DIR, STEP9_FILE)
    
MSV = 0.5 # Simplified expression value for marginally significant proteins

In [3]:
# Altair options
# alt.data_transformers.disable_max_rows()
# alt.data_transformers.enable('json')

## Function to plot top pathways across cancer types

In [4]:
def plot_top_ten(enrich_file_path, expr_file_path, xtitle):
    
    # Read in the expression data
    all_expression_data = pd.read_csv(expr_file_path, sep="\t", index_col=0)

    # Make a column where all increases are +1 and all decreases 
    # are -1, because these are ratioed abundances, so we can't 
    # compare magnitudes between genes--we can only compare whether 
    # there was a change, and whether it was positive or negative
    all_expression_data = all_expression_data.assign(simplified_change=np.nan)

    # adj p < 0.05 and change > 1 => +1
    all_expression_data.loc[
        (all_expression_data["change"] > 0) & (all_expression_data["adj_p"] < 0.05), 
        "simplified_change"
    ] = 1

    # adj p >= 0.05 and change > 1 => +0.5
    all_expression_data.loc[(all_expression_data["change"] > 0) & (all_expression_data["adj_p"] >= 0.05),
        "simplified_change"
    ] = MSV

    # change == 0 => 0
    all_expression_data.loc[
        all_expression_data["change"] == 0,
        "simplified_change"
    ] = 0

    # adj p >= 0.05 and change < 1 => -0.5
    all_expression_data.loc[(all_expression_data["change"] < 0) & (all_expression_data["adj_p"] >= 0.05), 
        "simplified_change"
    ] = -MSV

    # adj p < 0.05 and change < 1 => -1
    all_expression_data.loc[
        (all_expression_data["change"] < 0) & (all_expression_data["adj_p"] < 0.05),
        "simplified_change"
    ] = -1

    # Select just the proteins where we chose to reject the null hypothesis of no change
    expression_data = all_expression_data[all_expression_data["reject_null"]]

    # Read in the enrichment data
    enrichment_data = pd.read_csv(enrich_file_path, sep="\t")
    enrichment_data.sort_values(by=["cancer_type", "fdr"])

    # Assign pathway ranks within each cancer type based on FDR
    enrichment_data = enrichment_data.\
        assign(cancer_rank_fdr=enrichment_data.groupby("cancer_type")["fdr"].rank()).\
        sort_values(by=["cancer_type", "cancer_rank_fdr"]).\
        reset_index(drop=True)

    # Make a table with summary info for all pathways
    enrichment_summary = enrichment_data[["Term"]].drop_duplicates(keep="first")

    pathway_times_enriched = enrichment_summary["Term"].apply(
        lambda x: enrichment_data[enrichment_data["Term"] == x].shape[0])

    avg_rank = enrichment_summary["Term"].apply(
        lambda x: enrichment_data.loc[enrichment_data["Term"] == x, "cancer_rank_fdr"].mean())

    enrichment_summary = enrichment_summary.\
        assign(
            pathway_times_enriched=pathway_times_enriched,
            pathway_avg_rank=avg_rank).\
        sort_values(
            by=["pathway_times_enriched", "pathway_avg_rank"],
            ascending=[False, True]).\
        reset_index(drop=True)

    # Merge in the original enrichment data
    enrichment_data = enrichment_data.\
        merge(
            enrichment_summary,
            how="outer",
            left_on="Term",
            right_on="Term",
            validate="many_to_one"
        ).\
        sort_values(
            by=["pathway_times_enriched", "pathway_avg_rank", "cancer_type"],
            ascending=[False, True, True]
        )

    # Select top 10 for our plot
    top_ten = enrichment_data["Term"].unique()[:10]
    sel_enrich = enrichment_data[enrichment_data["Term"].isin(top_ten)]

    # Calculate the mean expression for each pathway in each cancer type
    mean_exprs = []

    for idx in sel_enrich.index:
        genes = sel_enrich.loc[idx, "genes"].split(";")
        cancer_type = sel_enrich.loc[idx, "cancer_type"]

        genes_expr = expression_data.loc[
            expression_data["protein_str"].isin(genes),
            "simplified_change"
        ].mean()

        mean_exprs.append(genes_expr)

    sel_enrich = sel_enrich.assign(mean_expr=mean_exprs)

    sel_enrich = sel_enrich.assign(
        rank_size=1 / sel_enrich["cancer_rank_fdr"],
        avg_rank_size=1 / sel_enrich["pathway_avg_rank"])

    individual = alt.Chart(sel_enrich).mark_circle().encode(
        x=alt.X(
            "Term:N",
            sort=sel_enrich["Term"].values,
            axis=alt.Axis(
                labelAngle=-30,
                labelFontSize=12,
                labelLimit=500,
                title="",
                titleFontSize=16
            )
        ),
        y=alt.Y(
            "cancer_type:N",
            axis=alt.Axis(
                title="Cancer type",
                titleFontSize=12
            ),
        ),
        size=alt.Size(
            "rank_size:Q",
            legend=None
        ),
        color=alt.Color(
            "mean_expr:Q",
            scale=alt.Scale(
                scheme="blueorange",
                domain=[-1, 1]
            ),
            legend=alt.Legend(
                title="Pathway tumor expression"
            )
        )
    ).properties(
        width=400,
        height=300
    )

    aggregate = alt.Chart(sel_enrich).mark_circle().encode(
        x=alt.X(
            "pathway_avg_rank:N",
            sort=sel_enrich["pathway_avg_rank"].values,
            axis=alt.Axis(
                labelAngle=-30,
                labelFontSize=12,
                labelLimit=500,
                title="Overall rank of pathway",
                titleFontSize=12
            )
        ),
        size=alt.Size(
            "avg_rank_size:Q",
            legend=None
        ),
    ).properties(
        width=400
    )

    full_plot = alt.vconcat(
        aggregate, individual
    ).properties(
        title=xtitle
    )
    
    return full_plot, enrichment_data, all_expression_data

In [5]:
alt.vconcat(
    plot_top_ten(STEP9_FILE_PATH, STEP1_FILE_PATH, "Reactome data - prerank")[0],
).configure_axis(
    grid=True
).configure_title(
    fontSize=16,
    anchor="end",
    offset=20
)

alt.VConcatChart(...)

In [ ]:
plot, enrich, epxr = plot_top_ten(STEP5_ABS_S2N_PATH, STEP1_FILE_PATH, "Reactome data - proteins ranked by |S2N|")

In [ ]:
enrich = enrich[enrich["pathway_times_enriched"] == 8]
ranks = enrich[["Term", "pathway_avg_rank"]].\
    set_index("Term").\
    drop_duplicates(keep="first").\
    rank()["pathway_avg_rank"].\
    rename("pathway_overall_rank")

enrich = enrich.merge(
    right=ranks,
    left_on="Term",
    right_on="Term",
    validate="many_to_one")

In [ ]:
enrich[enrich["Term"] == "Neutrophil degranulation"]

## Figure 2: Expression of proteins in the neutrophil degranulation pathway

In [ ]:
# Get the ID the top expessed pathway
neutro_pathway_ids = enrichment_data.\
    loc[enrichment_data["name"] == "Neutrophil degranulation", "pathway_id"].\
    unique()

if len(neutro_pathway_ids) != 1:
    raise ValueError("Multiple pathways found.")
    
neutro_pathway_id = neutro_pathway_ids[0]

### A: With proteins in different orders for the up and down categories

In [ ]:
def expr_heatmap(expr_data, enrich_data, pathway_id, up_or_down, max_p, x_title=True):
    """Make a heatmap with proteins on the X axis, cancer types on the Y 
    axis, and the color reflecting the expression of that protein in that
    cancer type.
    
    Parameters:
    expr_data (pandas.DataFrame): A dataframe with columns "cancer_type", "protein_str", "adj_p", and "simplified_change".
    enrich_data (pandas.DataFrame): A dataframe with columns "cancer_type" and "mean_expr" (i.e. mean expression for the pathway of interest).
    pathway_id (str): The Reactome pathway id for the pathway you want the heatmap for.
    up_or_down (str): Either "up" or "down"; whether to plot cancers where the pathway is up, or where it's down.
    max_p (float): The minimum p value for a protein to be included on the plot. If max_p > 0.05, then proteins with 0.05 <= p < max_p will be marked as marginally significant.
    x_title (bool): Whether to include an x axis title.
    
    Returns:
    altair.Chart: The heatmap.
    """
    # Filter expression data by p value
    expr_data = expr_data[expr_data["adj_p"] < max_p]
    
    # Get expression data for proteins in the pathway
    pathway_proteins = ut.search_reactome_proteins_in_pathways(pathway_id)
    pathway_expr = expr_data[expr_data["protein_str"].isin(pathway_proteins["member"])]
    
    # Categorize the cancer types as the neutrophil degranulation pathway's mean expression being up or down
    if up_or_down == "up":
        comparison = operator.gt
    elif up_or_down == "down":
        comparison = operator.lt
    else:
        raise ValueError(f"Invalid value for up_or_down parameter: '{up_or_down}'")
    
    sel_cancers = enrich_data.\
        loc[(enrich_data["pathway_id"] == neutro_pathway_id) & (comparison(enrich_data["mean_expr"], 0)),
           ["cancer_type", "mean_expr"]]

    # Select data for the specified cancer types
    selected_expr_data = pathway_expr[pathway_expr["cancer_type"].isin(sel_cancers["cancer_type"])]

    # Sort proteins by mean expression across selected cancer types
    prot_order = selected_expr_data.groupby("protein_str")[["simplified_change", "cancer_type"]].\
        agg({
            "simplified_change": np.mean,
            "cancer_type": "count"
        })

    prot_order = prot_order.assign(
            cancer_type=np.copysign(prot_order["cancer_type"], prot_order["simplified_change"])
        ).\
        sort_values(by=["simplified_change", "cancer_type"])

    # Sort cancer types by mean pathway expression
    cancer_order = sel_cancers.\
        sort_values(by="mean_expr", ascending=False).\
        loc[:, "cancer_type"]
    
    # Map simplified_change to legend values
    selected_expr_data = selected_expr_data.assign(
        tumor_up_down=selected_expr_data["simplified_change"].replace({
            1: "Up (adj p < 0.05)",
            MSV: f"Up (0.05 <= adj p < {max_p})",
            0: "No change",
            -MSV: f"Down (0.05 <= adj p < {max_p})",
            -1: "Down (adj p < 0.05)"
        })
    )
    
    # Plot
    plot = alt.Chart(selected_expr_data).mark_rect().encode(
        x=alt.X(
            "protein_str:N",
            sort=prot_order.index.values,
            axis=alt.Axis(
                labels=False,
                ticks=False,
                title=f"Proteins for {pathway_id} (different order in each facet)" if x_title else ""
            )
        ),
        y=alt.Y(
            "cancer_type:N",
            sort=cancer_order.values,
            axis=alt.Axis(
                title=f"Pathway {up_or_down} in tumor"
            )
        ),
        color=alt.Color(
            "tumor_up_down:N",
            scale=alt.Scale(
                domain=[
                    "Up (adj p < 0.05)",
                    f"Up (0.05 <= adj p < {max_p})",
                    "No change",
                    f"Down (0.05 <= adj p < {max_p})",
                    "Down (adj p < 0.05)"
                ],
                range=["#bf363a",
                       "#f6bda4",
                       "#f2efee",
                       "#afd3e6",
                       "#1e588a"
                ]
            ),
            legend=alt.Legend(
                title="Expression in tumor"
            )
        ),
    ).properties(
        width=700,
        height=175
    )
    
    return plot

In [ ]:
# Plot
alt.vconcat(
    expr_heatmap(all_expression_data, enrichment_data, neutro_pathway_id, "up", max_p=0.2, x_title=False),
    expr_heatmap(all_expression_data, enrichment_data, neutro_pathway_id, "down", max_p=0.2)
)

### B: With same sorting for both

In [ ]:
def expr_heatmap_alt_sort(expr_data, enrich_data, pathway_id, max_p):
    """Make a heatmap with proteins on the X axis, cancer types on the Y 
    axis, and the color reflecting the expression of that protein in that
    cancer type.
    
    Parameters:
    expr_data (pandas.DataFrame): A dataframe with columns "cancer_type", "protein_str", "adj_p", and "simplified_change".
    enrich_data (pandas.DataFrame): A dataframe with columns "cancer_type" and "mean_expr" (i.e. mean expression for the pathway of interest).
    pathway_id (str): The Reactome pathway id for the pathway you want the heatmap for.
    up_or_down (str): Either "up" or "down"; whether to plot cancers where the pathway is up, or where it's down.
    max_p (float): The minimum p value for a protein to be included on the plot. If max_p > 0.05, then proteins with 0.05 <= p < max_p will be marked as marginally significant.
    
    Returns:
    altair.Chart: The heatmap.
    """
    # Filter expression data by p value
    expr_data = expr_data[expr_data["adj_p"] < max_p]
    
    # Get expression data for proteins in the pathway
    pathway_proteins = ut.search_reactome_proteins_in_pathways(pathway_id)
    pathway_expr = expr_data[expr_data["protein_str"].isin(pathway_proteins["member"])]
    
    # Sort proteins by mean expression across all cancer types
    prot_order = pathway_expr.groupby("protein_str")[["simplified_change", "cancer_type"]].\
        agg({
            "simplified_change": np.mean,
            "cancer_type": "count"
        })

    prot_order = prot_order.assign(
            cancer_type=np.copysign(prot_order["cancer_type"], prot_order["simplified_change"])
        ).\
        sort_values(by=["simplified_change", "cancer_type"])

    # Sort cancer types by mean pathway expression
    sel_cancers = enrich_data.\
        loc[enrich_data["pathway_id"] == neutro_pathway_id, ["cancer_type", "mean_expr"]]
        
    sel_cancers = sel_cancers.\
        assign(up_or_down=np.where(sel_cancers["mean_expr"] > 0, "Pathway up in tumor", "Pathway down in tumor")).\
        sort_values(by="mean_expr", ascending=False).\
        loc[:, ["cancer_type", "up_or_down"]]
    
    cancer_order = sel_cancers["cancer_type"].values
    
    pathway_expr = pathway_expr.merge(
        right=sel_cancers,
        how="inner",
        left_on="cancer_type",
        right_on="cancer_type",
        validate="many_to_one"
    )
    
    # Map simplified_change to legend values
    pathway_expr = pathway_expr.assign(
        tumor_up_down=pathway_expr["simplified_change"].replace({
            1: "Up (adj p < 0.05)",
            MSV: f"Up (0.05 <= adj p < {max_p})",
            0: "No change",
            -MSV: f"Down (0.05 <= adj p < {max_p})",
            -1: "Down (adj p < 0.05)"
        })
    )
    
    # Plot
    plot = alt.Chart(pathway_expr).mark_rect().encode(
        x=alt.X(
            "protein_str:N",
            sort=prot_order.index.values,
            axis=alt.Axis(
                labels=False,
                ticks=False,
                title=f"Proteins for {pathway_id} (same order in both facets)"
            )
        ),
        y=alt.Y(
            "cancer_type:N",
            sort=cancer_order,
            axis=alt.Axis(
                title=""
            )
        ),
        color=alt.Color(
            "tumor_up_down:N",
            scale=alt.Scale(
                domain=[
                    "Up (adj p < 0.05)",
                    f"Up (0.05 <= adj p < {max_p})",
                    "No change",
                    f"Down (0.05 <= adj p < {max_p})",
                    "Down (adj p < 0.05)"
                ],
                range=["#bf363a",
                       "#f6bda4",
                       "#f2efee",
                       "#afd3e6",
                       "#1e588a"
                ]
            ),
            legend=alt.Legend(
                title="Expression in tumor"
            )
        ),
        row=alt.Row(
            "up_or_down:N",
            sort="descending",
            title="",
            header=alt.Header(
                labelFontStyle="bold"
            )
        )
    ).properties(
        width=600,
        height=150
    ).resolve_scale(
        y="independent" # This makes it so each facet (up or down) only has the cancers that are in that category
    )
    
    return plot

In [ ]:
# Plot
expr_heatmap_alt_sort(all_expression_data, enrichment_data, neutro_pathway_id, max_p=0.2)

## Figure 3: Pathway overlay for neutrophil degranulation

In [ ]:
def pathway_overlay_wrapper(pathway_id, exp_data, enrich_data, up_or_down):
    
    prots = ut.search_reactome_proteins_in_pathways(pathway_id)
    
    # Categorize the cancer types as the neutrophil degranulation pathway's mean expression being up or down
    if up_or_down == "up":
        comparison = operator.gt
    elif up_or_down == "down":
        comparison = operator.lt
    else:
        raise ValueError(f"Invalid value for up_or_down parameter: '{up_or_down}'")
    
    sel_cancers = enrich_data.\
        loc[(enrich_data["pathway_id"] == neutro_pathway_id) & (comparison(enrich_data["mean_expr"], 0)),
            "cancer_type"]
    
    # Select the desired expression data and average the simplified change across cancer types
    sel_exp = exp_data.\
        loc[
            exp_data["protein_str"].isin(prots["member"]) & 
            exp_data["cancer_type"].isin(sel_cancers),
            ["protein_str", "simplified_change"]
        ].\
        groupby("protein_str").\
        agg(np.mean).\
        rename(columns={"simplified_change": f"{up_or_down}_simp_change"})
    
    img_name = f"{TIME_START}_{pathway_id}_{up_or_down}.png"
    img_path = os.path.join(STEP7_DIR, img_name)
    
    token, _ = ut.reactome_enrichment_analysis(
        analysis_type="ranked", 
        data=sel_exp, 
        sort_by="ENTITIES_FDR", 
        ascending=True,
        include_high_level_diagrams=True, 
        include_interactors=False)
    
    _, img_path = ut.reactome_pathway_overlay(
        analysis_token=token,
        pathway=pathway_id,
        open_browser=False,
        export_path=img_path,
        image_format="png",
        display_col_idx=None,
        diagram_colors="Modern",
        overlay_colors="Standard",
        quality=10
    )

    _, url = ut.reactome_pathway_overlay(
        analysis_token=token,
        pathway=pathway_id,
        open_browser=False,
    )

    return img_path, url

In [ ]:
# Note that we're using expression_data, not all_expression_data; the former has
# only proteins where reject_null

up_image_path, up_url = pathway_overlay_wrapper(neutro_pathway_id, expression_data, enrichment_data, "up")

In [ ]:
IPython.display.HTML(f'<a href={up_url}>{up_url}</a>')

In [ ]:
IPython.display.Image(up_image_path)

In [ ]:
# Note that we're using expression_data, not all_expression_data; the former has
# only proteins where reject_null

down_image_path, down_url = pathway_overlay_wrapper(neutro_pathway_id, expression_data, enrichment_data, "down")

In [ ]:
IPython.display.HTML(f'<a href={down_url}>{down_url}</a>')

In [ ]:
IPython.display.Image(down_image_path)

## Figure 4: Distribution of tumor stage in each cancer
Since neutrophil degranulation is associated with tumor progression, we want to check that it's not just up in cohorts with more high stage tumors, and down in cohorts with more low stage tumors.

In [ ]:
cc = cptac.Ccrcc(no_internet=True)
co = cptac.Colon(no_internet=True)
en = cptac.Endometrial(no_internet=True)
gb = cptac.Gbm(no_internet=True)
hn = cptac.Hnscc(no_internet=True)
ls = cptac.Lscc(no_internet=True)
lu = cptac.Luad(no_internet=True)
ov = cptac.Ovarian(no_internet=True)

In [ ]:
stage_df = pd.DataFrame()

In [ ]:
cc_stage = cc.get_clinical()["tumor_stage_pathological"].\
    dropna().\
    str.rsplit(" ", n=1, expand=True)[1].\
    rename("tumor_stage").\
    reset_index(drop=False).\
    assign(cancer_type="ccrcc")

stage_df = stage_df.append(cc_stage)

In [ ]:
co_stage = co.get_clinical()["Stage"].\
    dropna().\
    str.rsplit(" ", n=1, expand=True)[1].\
    rename("tumor_stage").\
    reset_index(drop=False).\
    assign(cancer_type="colon")

stage_df = stage_df.append(co_stage)

In [ ]:
en_stage = en.get_clinical()["tumor_Stage-Pathological"].\
    dropna().\
    str.rsplit(" ", n=1, expand=True)[1].\
    rename("tumor_stage").\
    reset_index(drop=False).\
    assign(cancer_type="endometrial")

stage_df = stage_df.append(en_stage)

In [ ]:
# GBM: All tumors are stage IV
gb_stage = pd.Series("IV", gb.get_clinical().index).\
    rename("tumor_stage").\
    reset_index(drop=False).\
    assign(cancer_type="gbm")

gb_stage
stage_df = stage_df.append(gb_stage)

In [ ]:
hn_stage = hn.get_clinical()["patho_staging_curated"].\
    dropna().\
    str.rsplit(" ", n=1, expand=True)[1].\
    rename("tumor_stage").\
    reset_index(drop=False).\
    assign(cancer_type="hnscc")

stage_df = stage_df.append(hn_stage)

In [ ]:
ls_stage = ls.get_clinical()["Stage"]
ls_stage = ls_stage.\
    rename("tumor_stage").\
    where(
        cond=~(ls_stage.dropna().str.endswith("A") | ls_stage.dropna().str.endswith("B")),
        other=ls_stage.str[:-1]).\
    where(
        cond=~(ls_stage.dropna().str.endswith("A 3")),
        other=ls_stage.dropna().str[:-3]
    ).\
    reset_index(drop=False).\
    assign(cancer_type="lscc")

stage_df = stage_df.append(ls_stage)

In [ ]:
lu_stage = lu.get_clinical()["Stage"]
lu_stage = lu_stage.\
    rename("tumor_stage").\
    where(
        cond=~(lu_stage.dropna().str.endswith("A") | lu_stage.dropna().str.endswith("B")),
        other=lu_stage.str[:-1]).\
    replace({
        "1": "I",
        "2": "II",
        "3": "III"}).\
    reset_index(drop=False).\
    assign(cancer_type="luad")

stage_df = stage_df.append(lu_stage)

In [ ]:
ov_stage = ov.get_clinical()["Tumor_Stage_Ovary_FIGO"]
ov_stage = ov_stage.\
    rename("tumor_stage").\
    where(
        cond=~(ov_stage.dropna().str.endswith("A") | 
               ov_stage.dropna().str.endswith("B") | 
               ov_stage.dropna().str.endswith("C")),
        other=ov_stage.str[:-1]).\
    drop(ov_stage[ov_stage == "Not Reported/ Unknown"].index).\
    reset_index(drop=False).\
    assign(cancer_type="ovarian")

stage_df = stage_df.append(ov_stage)

In [ ]:
# Drop NaN samples
stage_df = stage_df.dropna()

# Drop normal samples (to avoid duplicate counts)
stage_df = stage_df[~stage_df["Patient_ID"].str.endswith(".N")]

In [ ]:
# Join in whether neutrophil expression is up or down in tumor
neutro_expr = enrichment_data.loc[
    enrichment_data["pathway_id"] == neutro_pathway_id, 
    ["cancer_type", "name", "pathway_id", "mean_expr"]]

neutro_expr

In [ ]:
stage_df = stage_df.merge(
    right=neutro_expr,
    how="left",
    left_on="cancer_type",
    right_on="cancer_type",
    validate="many_to_one")

In [ ]:
# Make plots
alt.Chart(stage_df).mark_bar().encode(
    x=alt.X(
        "tumor_stage:N",
        axis=alt.Axis(
            labelAngle=0,
            labelFontSize=14,
            title=""
        )
    ),
    y=alt.Y(
        "count():Q",
        axis=alt.Axis(
            title="Count",
            titleFontSize=12
        )
    ),
    color=alt.Color(
        "mean_expr:Q",
        scale=alt.Scale(
            scheme="blueorange",
            domain=[-1, 1]
        ),
        legend=alt.Legend(
            title=["Neutrophil degranulation:", "Mean pathway expression", "in tumor"]
        )
    ),
    column=alt.Column(
        "cancer_type:N",
        title="Tumor stage distributions",
    )
).properties(
    width=65
).configure_header(
    titleFontSize=12
)

In [ ]:
# Ask Sam: Is this idea of "mean pathway expression" 
# really valid? What if in reality, though we can't
# measure it, some proteins' magnitude of expression 
# change is a lot greater, so they should actually
# have greater weight in the mean calculation, and
# thus might alter the overall statistic for the
# pathway?